# Working with MERFISH data

This tutorial reads MERFISH data into stLearn, uses the blank control barcodes for quality control, integrates three replicates and clusters the cells.

The data are from [Xia et al. (2019), *PNAS* 116(39):19490](https://doi.org/10.1073/pnas.1912459116): MERFISH profiling of 10,050 genes in cultured U-2 OS cells, imaged in three replicates. We use two of the paper's supplementary files:

- `pnas.1912459116.sd12.csv`: the count matrix, genes × cells, for all three replicates
- `pnas.1912459116.sd15.xlsx`: the cell positions in microns, one sheet per replicate

The original PNAS download links no longer work, so we download the same files from the copy kept in the Giotto team's [spatial-datasets](https://github.com/drieslab/spatial-datasets) repository on GitHub.

In [ ]:
import harmonypy as hm
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import stlearn as st

# Setting figure parameters
st.settings.set_figure_params(dpi=120)

## Download the data

In [ ]:
import urllib.request
from pathlib import Path

base_url = (
    "https://raw.githubusercontent.com/drieslab/spatial-datasets/master/"
    "data/2019_merFISH_PNAS/raw_data/"
)
data_dir = Path("data")
data_dir.mkdir(exist_ok=True)

count_file = data_dir / "pnas.1912459116.sd12.csv"
spatial_file = data_dir / "pnas.1912459116.sd15.xlsx"

for path in (count_file, spatial_file):
    if not path.exists():
        urllib.request.urlretrieve(base_url + path.name, path)

## Read the data

With `sheet_name=None`, `st.io.read_merfish` reads every sheet of positions, one per replicate, lays the replicates out side by side and records each cell's replicate in `obs['batch']`. Positions stay in microns, so distances keep their real size.

MERFISH gives each gene a binary barcode. Of the 12,903 possible barcodes, 2,853 were not assigned to any gene. Counts on these blank barcodes are decoding errors, so `read_merfish` keeps them out of the expression matrix: they are in `obsm['blank']`, with each cell's blank total in `obs['blank_counts']` and its share of all the cell's counts in `obs['blank_fraction']`.

In [ ]:
data = st.io.read_merfish(count_file, spatial_file, sheet_name=None)
data

In [ ]:
data.obs["batch"].value_counts(sort=False)

## Blank controls

MERFISH gives each gene a binary barcode. Of the 12,903 possible barcodes, 2,853 were not assigned to any gene. Any counts on these blank barcodes are decoding errors, so they measure how often a barcode is misread. The paper reports a misidentification rate of 3.9%.

In [ ]:
blank_means = data.obsm["blank"].mean(axis=0)
gene_means = np.asarray(data.X.mean(axis=0)).ravel()

print(f"{len(blank_means):,} blank barcodes, {len(gene_means):,} genes")
print(f"Misidentification rate: {blank_means.mean() / gene_means.mean():.1%}")

Following the paper, we keep the genes whose average count per cell is at least the 95th percentile of the blank barcodes' averages: a gene detected less often than that can't be told apart from decoding errors. The blanks themselves are removed from the matrix.

In [ ]:
threshold = np.percentile(blank_means, 95)
expressed = gene_means >= threshold
data = data[:, expressed].copy()

print(f"Threshold: {threshold:.2f} counts per cell")
print(f"Kept {expressed.sum():,} of {len(expressed):,} genes")

## Quality check

Each dot is a cell, sized by its total counts and coloured by the number of genes detected.

In [ ]:
plt.rcParams["figure.figsize"] = (12, 4)
st.pl.qc_plot(data, name="merfish_qc")

In [ ]:
# The fraction of each cell's counts on blank barcodes is low in every replicate.
data.obs.groupby("batch")["blank_fraction"].describe()[["mean", "min", "max"]]

## Preprocessing

Normalise and log-transform, then keep the 1,600 most variable genes, similar to the 1,598 the paper used. With `batch_key`, the genes are chosen within each replicate, so differences between replicates don't drive the choice.

In [ ]:
st.pp.normalize_total(data)
st.pp.log1p(data)
st.pp.highly_variable_genes(data, n_top_genes=1600, batch_key="batch", subset=True)
st.pp.scale(data)

## PCA and integration with Harmony

Run PCA with 30 components, as in the paper, then correct them for differences between replicates with [Harmony](https://github.com/slowkow/harmonypy), as in the *Integrating Multiple Datasets* tutorial.

In [ ]:
st.em.run_pca(data, n_comps=30, random_state=0)

ho = hm.run_harmony(
    np.ascontiguousarray(data.obsm["X_pca"]), data.obs, "batch", verbose=False
)
data.obsm["X_pca"] = ho.Z_corr

## Leiden clustering

In [ ]:
st.pp.neighbors(data,n_neighbors=25)

In [ ]:
st.tl.clustering.leiden(data, resolution=0.5)

In [ ]:
pd.crosstab(data.obs["leiden"], data.obs["batch"])

In [ ]:
st.pl.cluster_plot(data, use_label="leiden", size=5)